In [22]:
import pandas as pd
import numpy as np
train=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\House_Price_Prediction\Kaggles_data\train.csv")
test=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\House_Price_Prediction\Kaggles_data\test.csv")
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [23]:
print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [24]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [25]:
#Remove outliers
train=train.drop(train[(train['GrLivArea']>4000) 
                 &(train['SalePrice']<300000)].index)
train=train.drop(train[(train['OverallQual']<5) 
                 &(train['SalePrice']>200000)].index)


In [26]:
#identify target and features
y=np.log1p(train['SalePrice'])
train=train.drop('SalePrice',axis=1)

In [27]:
#combine train+test
full=pd.concat([train,test],axis=0).reset_index(drop=True)

# Feature Engineering

In [28]:
#total Square feet
full['TotalSf']=(
    full['TotalBsmtSF']+full['1stFlrSF']+full['2ndFlrSF']
)

In [29]:
#Total Bathrooms
full['TotalBath']=(
    full['FullBath']+0.5*full['HalfBath']+full['BsmtFullBath']+0.5*full['BsmtHalfBath']
)

In [30]:
#Age features
full['HouseAge']=full['YrSold']-full['YearBuilt']
full['RemodAge']=full['YrSold']-full['YearRemodAdd']

In [31]:
#Binary Features
full['HasGarage']=(full['GarageArea']>0).astype(int)
full['HasBsmt']=(full['TotalBsmtSF']>0).astype(int)
full['HasPool']=(full['PoolArea']>0).astype(int)

In [32]:
#Porch Area
full['TotalPorch']=(
    full['OpenPorchSF']+
    full['EnclosedPorch']+
    full['3SsnPorch']+
    full['ScreenPorch']
)

In [33]:
#Interaction Features
full['OverallScore']=full['OverallQual']*full['OverallCond']
full['QualArea']=full['OverallQual']*full['GrLivArea']

In [34]:
#Handle missing values
for col in full.select_dtypes(include=[np.number]).columns:
    full[col]=full[col].fillna(full[col].median())
for col in full.select_dtypes(include=['object']).columns:
    full[col]=full[col].fillna("None")

In [35]:
from sklearn.model_selection import KFold,cross_val_score
from scipy.stats import skew

In [36]:
#Fix Skewed Numerical Features
numeric_f=full.select_dtypes(include=[np.number]).columns
skewed=full[numeric_f].apply(lambda x: skew(x)).sort_values(ascending=False)
skewed=skewed[skewed>0.75].index
full[skewed]=np.log1p(full[skewed])

In [37]:
#One Hot Encode
full=pd.get_dummies(full)

In [38]:
#Split Back
X=full.iloc[:len(train)]
X_test=full.iloc[len(train):]

In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold,cross_val_score
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor


In [40]:
model=LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.01,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgb_scores=cross_val_score(
    model,
    X,
    y,
    scoring='neg_root_mean_squared_error',
    cv=5
)
print("Fold RMSE scores : ")
print(-lgb_scores)
print("Mean CV RMSE: ")
print(-lgb_scores.mean())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002468 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4394
[LightGBM] [Info] Number of data points in the train set: 1165, number of used features: 203
[LightGBM] [Info] Start training from score 12.020981
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005421 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4410
[LightGBM] [Info] Number of data points in the train set: 1165, number of used features: 201
[LightGBM] [Info] Start training from score 12.022831
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM]

In [41]:
print(train.shape)
print(y.shape)
print(X.shape)
print(full.shape)
print("SalePrice"in X.columns)

(1457, 80)
(1457,)
(1457, 320)
(2916, 320)
False


In [42]:
from xgboost import XGBRegressor
xgb_model=XGBRegressor(
    n_estimators=3000,
    random_state=42,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=1,
    n_jobs=-1
)
xgb_scores=cross_val_score(
    xgb_model,X,y,
    scoring='neg_root_mean_squared_error',
    cv=5
)
xgb_rmse=-xgb_scores.mean()
print('XGB CV',xgb_rmse)


XGB CV 0.1118614021571723


In [43]:
'''from xgboost import XGBRegressor
xgb_model=XGBRegressor(
    n_estimators=5000,
    random_state=42,
    learning_rate=0.01,
    max_depth=4,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1,
    n_jobs=-1
)
xgb_scores=cross_val_score(
    xgb_model,X,y,
    scoring='neg_root_mean_squared_error',
    cv=5
)
xgb_rmse=-xgb_scores.mean()
print('Improved XGB CV',xgb_rmse)'''


"from xgboost import XGBRegressor\nxgb_model=XGBRegressor(\n    n_estimators=5000,\n    random_state=42,\n    learning_rate=0.01,\n    max_depth=4,\n    min_child_weight=1,\n    subsample=0.8,\n    colsample_bytree=0.8,\n    reg_alpha=0.05,\n    reg_lambda=1,\n    n_jobs=-1\n)\nxgb_scores=cross_val_score(\n    xgb_model,X,y,\n    scoring='neg_root_mean_squared_error',\n    cv=5\n)\nxgb_rmse=-xgb_scores.mean()\nprint('Improved XGB CV',xgb_rmse)"

In [44]:
import numpy as np
from sklearn.metrics import mean_squared_error
kf=KFold(n_splits=5,shuffle=True,random_state=42)
blend_rmse_list=[]
for train_idx,val_idx in kf.split(X):
    X_train,X_val=X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val=y.iloc[train_idx],y.iloc[val_idx]
    
    model.fit(X_train,y_train)
    xgb_model.fit(X_train,y_train)

    lgb_pred=model.predict(X_val)
    xgb_pred=xgb_model.predict(X_val)
    
    blend_pred=0.5*lgb_pred+0.5*xgb_pred
    rmse=np.sqrt(mean_squared_error(y_val,blend_pred))
    blend_rmse_list.append(rmse)

blend_rmse=np.mean(blend_rmse_list)
print("Blended CV : ",blend_rmse)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4421
[LightGBM] [Info] Number of data points in the train set: 1165, number of used features: 205
[LightGBM] [Info] Start training from score 12.028735
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003243 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4405
[LightGBM] [Info] Number of data points in the train set: 1165, number of used features: 202
[LightGBM] [Info] Start training from score 12.032415
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM]

In [45]:
#Add Ridge
from sklearn.linear_model import Ridge
ridge=Ridge(alpha=15)
scores=cross_val_score(
    ridge,
    X,
    y,
    scoring='neg_root_mean_squared_error',
    cv=5
)
print("Ridge Cv : ",-scores.mean())

Ridge Cv :  0.11294526827912452


In [46]:
import numpy as np
from sklearn.metrics import mean_squared_error
kf=KFold(n_splits=5,shuffle=True,random_state=42)
rmse_list=[]
for train_idx,val_idx in kf.split(X):
    X_train,X_val=X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val=y.iloc[train_idx],y.iloc[val_idx]
    
    xgb_model.fit(X_train,y_train)
    ridge.fit(X_train,y_train)

    xgb_pred=xgb_model.predict(X_val)
    ridge_pred=ridge.predict(X_val)
    
    blend_pred=0.8*xgb_pred+0.2*ridge_pred
    rmse=np.sqrt(mean_squared_error(y_val,blend_pred))
    rmse_list.append(rmse)

blend_rmse=np.mean(rmse_list)
print("Blended(XGB+Ridge) CV : ",blend_rmse)

Blended(XGB+Ridge) CV :  0.1117597700293584


In [49]:
#Create submission file
xgb_model.fit(X,y)
test_pred_log=xgb_model.predict(X_test)
test_pred=np.expm1(test_pred_log)

submission = pd.DataFrame({
    'ID':test['Id'],
    'SalePrice':test_pred
})
submission.to_csv("submission3.csv",index=False)